## Importações

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import pickle
import os

centralities = ["central", "peripheral"]
years = range(2014, 2025)

portfolios = {}
market_returns_dict = {}

k_values = [5]

for year in years:
    for k in k_values:
        full_ret = pd.read_parquet(f"../../data/02_clean/returns_new_{year-k+1}_{year}.parquet")
            
        market_returns_dict[f"{year}_{k}"] = np.log1p(full_ret).mean(axis=1)
        
        for centrality in centralities:
            cols = pd.read_csv(f"../../data/06_portfolios/{centrality}_{year}_{k}.csv", index_col="Date").columns.tolist()
            valid_cols = [c for c in cols if c in full_ret.columns]
            portfolios[f"{centrality}_{year}_{k}"] = full_ret[valid_cols]

/home/maria-eduarda/Área de trabalho/paper-financial-graph/.venv/lib/python3.14/site-packages/pandas/core/internals/blocks.py:395: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)
/home/maria-eduarda/Área de trabalho/paper-financial-graph/.venv/lib/python3.14/site-packages/pandas/core/internals/blocks.py:395: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)
/home/maria-eduarda/Área de trabalho/paper-financial-graph/.venv/lib/python3.14/site-packages/pandas/core/internals/blocks.py:395: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)
/home/maria-eduarda/Área de trabalho/paper-financial-graph/.venv/lib/python3.14/site-packages/pandas/core/internals/blocks.py:395: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)
/home/maria-eduarda/Área de trabalho/paper-financial-graph/.venv/lib/python3.14/site-packages/pandas/core/internals/

In [13]:
weekly_returns = {}
weekly2_returns = {}
monthly_returns = {}
daily_returns = {}

weighting = "equal"

df_mcap = pd.read_csv("../../data/02_clean/METADADOS_ATUALIZADO - Sheet1.csv")

for name, portfolio in portfolios.items():
    year = int(name.split('_')[1])
    k = int(name.split('_')[2])
    
    log_returns = np.log1p(portfolio)
    R_m = market_returns_dict[f"{year}_{k}"]
    
    mcap_col = f"mcap_{year}"
    if mcap_col not in df_mcap.columns:
        mcap_col = [c for c in df_mcap.columns if "mcap_" in c][-1]
        
    tickers = portfolio.columns.tolist()
    mcaps = df_mcap[df_mcap["Ticker"].isin(tickers)][["Ticker", mcap_col]].copy()
    mcaps[mcap_col] = pd.to_numeric(
        mcaps[mcap_col].astype(str).str.replace(',', '.'), 
        errors='coerce'
    ).fillna(0)

    if weighting == "mcap": 
        weights_df = pd.DataFrame({"Ticker": tickers}).merge(mcaps, on="Ticker", how="left").fillna(0)
        w = weights_df[mcap_col].values
        if w.sum() == 0:
            w = np.ones(len(w)) / len(w)
        else:
            w = w / w.sum()
            
        vw_returns = (log_returns * w).sum(axis=1)
    elif weighting == "equal":    
        vw_returns = (log_returns).mean(axis=1)
    
    # Calculate excess returns
    daily_returns[name] = vw_returns #- R_m
    
    weekly_m = R_m.resample("W-FRI").sum()
    weekly_returns[name] = vw_returns.resample("W-FRI").sum() #- weekly_m
    
    weekly2_m = R_m.resample("2W-FRI").sum()
    weekly2_returns[name] = vw_returns.resample("2W-FRI").sum() #- weekly2_m
    
    monthly_m = R_m.resample("ME").sum()
    monthly_returns[name] = vw_returns.resample("W-FRI").sum() #- monthly_m

/home/maria-eduarda/Área de trabalho/paper-financial-graph/.venv/lib/python3.14/site-packages/pandas/core/internals/blocks.py:395: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)


## Lead-lag

In [11]:
def autocorrelation_matrix(X, lag):
    X_t = X.iloc[lag:]
    X_tk = X.shift(lag).iloc[lag:]

    mu_t = X_t.mean().values
    mu_tk = X_tk.mean().values

    Xc_t = X_t.values - mu_t
    Xc_tk = X_tk.values - mu_tk

    Sigma_k = (Xc_tk.T @ Xc_t) / len(Xc_t)

    std_t = X_t.std(ddof=0).values
    std_tk = X_tk.std(ddof=0).values

    # Avoid division by zero
    std_t[std_t == 0] = 1e-8
    std_tk[std_tk == 0] = 1e-8

    return Sigma_k / np.outer(std_tk, std_t)

def plot_antisymmetric_autocorr(
    returns_list,
    column_names,
    labels,
    lags=(1, 2, 3, 4),
    figsize=(12, 8),
    cmap="Blues",
    title_prefix="Y",
    annot=False,
    diff=True,
    fig_title=None
):
    X = pd.concat(returns_list, axis=1).dropna()
    X.columns = column_names

    fig, axes = plt.subplots(2, 2, figsize=figsize)
    axes = axes.flatten()

    if len(lags) == 1:
        axes = [axes]

    for ax, lag in zip(axes, lags):
        T = autocorrelation_matrix(X, lag)

        if diff:
            A = pd.DataFrame(
                T - T.T,
                index=labels,
                columns=labels
            )
            title = f"{title_prefix}({lag}) - {title_prefix}'({lag})"
        else:
            A = pd.DataFrame(
                T,
                index=labels,
                columns=labels
            )
            title = f"{title_prefix}({lag})"

        sns.heatmap(A, ax=ax, cmap=cmap, center=0, annot=annot)
        ax.set_title(title)

    if fig_title:
        fig.suptitle(fig_title, fontsize=14)
    plt.tight_layout()
    plt.show()

### Central Peripheral

In [15]:
years = range(2014, 2025)

for year in years:
    for k in k_values:
        try:
            R1 = daily_returns[f"peripheral_{year}_{k}"]
            R2 = daily_returns[f"central_{year}_{k}"]
            R3 = weekly_returns[f"peripheral_{year}_{k}"]
            R4 = weekly_returns[f"central_{year}_{k}"]
            R5 = weekly2_returns[f"peripheral_{year}_{k}"]
            R6 = weekly2_returns[f"central_{year}_{k}"]
            R7 = monthly_returns[f"peripheral_{year}_{k}"]
            R8 = monthly_returns[f"central_{year}_{k}"]

        except KeyError as e:
            print(f"Missing data for {year}: {e}")

        # =============================
        # 5. Lead–lag matrix
        # =============================
        matrix = []

        pairs = [(R1, R2), (R3, R4), (R5, R6), (R7, R8)]
        max_lag = 5

        for R_peri, R_cent in pairs:
            X = pd.concat([R_peri, R_cent], axis=1).dropna()
            X.columns = ["Peripheral", "Central"]
            
            valid_lags = [l for l in range(1, max_lag + 1) if len(X) >= l + 4]
            if not valid_lags:
                matrix.append(float('nan'))
                continue
                
            total_lead_lag = 0
            for l in valid_lags:
                acm = autocorrelation_matrix(X, lag=l)
                lag_matrix = acm - acm.T
                total_lead_lag += lag_matrix[1, 0]
            
            matrix.append(total_lead_lag)

        # =============================
        # 6. DataFrame final
        # =============================
        cols = ["cp1d", "cp1w", "cp2w", "cp1m"]

        leadlag_df = pd.DataFrame(
            [matrix],
            columns=cols,
            index=[year]
        )

        leadlag_df.to_csv(
            f"../../data/08_lead_lag/leadlag_df_{year}_{k}.csv"
        )


In [5]:
matrix

[np.float64(-0.05604789489601001),
 np.float64(0.0519669191001194),
 np.float64(0.07077582111634129),
 np.float64(0.17634963263070041)]

### Lead-lag considerando Market Cap

In [ ]:
# import pandas as pd
# import numpy as np

In [47]:
df_mcap = pd.read_csv(
    "../../data/02_clean/METADADOS_ATUALIZADO - Sheet1.csv"
)

In [9]:
years = range(2015, 2025)

# # Load metadata and returns once outside the loop for efficiency
df_mcap_meta = pd.read_csv(
    "../../data/02_clean/METADADOS_ATUALIZADO - Sheet1.csv"
)
df_ret_full = pd.read_parquet("../../data/02_clean/returns_30_years.parquet")
df_ret_full = df_ret_full[df_ret_full.index >= pd.Timestamp(2015, 1, 1)]

for year in years:
    try:
        # 1. Load returns for the calculation window (e.g., year-9 to year)
        returns_prev = pd.read_parquet(f"../../data/02_clean/returns_new_{year-9}_{year}.parquet")
        
        # 2. Market cap cleaning and quantile logic
        df_mcap_year = df_mcap_meta[(df_mcap_meta["Ticker"].isin(returns_prev.columns)) &
        (df_mcap_meta["Ticker"].isin(df_ret_full.columns))].copy()
        mcap_col = f"mcap_{year}"

        # Clean numeric data (handle dots and commas)
        df_mcap_year[mcap_col] = (
            df_mcap_year[mcap_col]
            .replace("#ERROR!", np.nan)
            .astype(str)
            .str.replace(".", "", regex=False)
            .str.replace(",", ".", regex=False)
        )
        df_mcap_year[mcap_col] = pd.to_numeric(df_mcap_year[mcap_col], errors="coerce")
        df_mcap_year = df_mcap_year.dropna(subset=[mcap_col])

        # Define Portfolios based on p80/p20 quantiles
        p80 = df_mcap_year[mcap_col].quantile(0.8)
        p20 = df_mcap_year[mcap_col].quantile(0.2)

        large_tickers = df_mcap_year[df_mcap_year[mcap_col] >= p80]["Ticker"].tolist()
        small_tickers = df_mcap_year[df_mcap_year[mcap_col] <= p20]["Ticker"].tolist()

        # 3. Calculate Portfolio Returns (Log returns)
        # R1/R2: Daily
        log_rets = np.log1p(returns_prev)
        R1 = log_rets[large_tickers].mean(axis=1)
        R2 = log_rets[small_tickers].mean(axis=1)

        # R3/R4: Weekly (Friday)
        R3 = log_rets[large_tickers].resample("W-FRI").sum().mean(axis=1)
        R4 = log_rets[small_tickers].resample("W-FRI").sum().mean(axis=1)

        # R5/R6: 2-Week
        R5 = log_rets[large_tickers].resample("2W-FRI").sum().mean(axis=1)
        R6 = log_rets[small_tickers].resample("2W-FRI").sum().mean(axis=1)

        # R7/R8: Monthly
        R7 = log_rets[large_tickers].resample("M").sum().mean(axis=1)
        R8 = log_rets[small_tickers].resample("M").sum().mean(axis=1)

        # 4. Lead-Lag Matrix Calculation
        matrix = []
        lag = 1
        
        # Pairs to iterate through (Small vs Large at different frequencies)
        pairs = [(R1, R2), (R3, R4), (R5, R6), (R7, R8)]
        
        for s_ret, l_ret in pairs:
            X = pd.concat([s_ret, l_ret], axis=1).dropna()
            X.columns = ["Large", "Small"]
            
            acm = autocorrelation_matrix(X, lag)
            # Cross-autocorrelation asymmetry: (Large leads Small) - (Small leads Large)
            lag_matrix = acm - acm.T
            matrix.append(lag_matrix[1, 0])

        # 5. Save Results
        cols = ["ls1d", "ls1w", "ls2w", "ls1m"]
        leadlag_df = pd.DataFrame([matrix], columns=cols, index=[year])
        leadlag_df.to_csv(f"../../data/08_lead_lag/marketcap_leadlag_df_{year}.csv")
        
    except Exception as e:
        print(f"Error processing year {year}: {e}")
        continue

/tmp/ipykernel_7852/767302368.py:53: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  R7 = log_rets[large_tickers].resample("M").sum().mean(axis=1)
/tmp/ipykernel_7852/767302368.py:54: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  R8 = log_rets[small_tickers].resample("M").sum().mean(axis=1)
/tmp/ipykernel_7852/767302368.py:53: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  R7 = log_rets[large_tickers].resample("M").sum().mean(axis=1)
/tmp/ipykernel_7852/767302368.py:54: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  R8 = log_rets[small_tickers].resample("M").sum().mean(axis=1)
/tmp/ipykernel_7852/767302368.py:53: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  R7 = log_rets[large_tickers].resample("M").sum().mean(axis=1)
